In [ ]:
import os
WORKSHOP_RESOURCE_GROUP = os.getenv("WORKSHOP_RESOURCE_GROUP", os.getenv("RESOURCE_GROUP_NAME", "YOUR_RESOURCE_GROUP")).strip()
WORKSHOP_AUTH_MODE = os.getenv("WORKSHOP_AUTH_MODE", "managed-identity").strip()
os.environ["WORKSHOP_RESOURCE_GROUP"] = WORKSHOP_RESOURCE_GROUP
os.environ["WORKSHOP_AUTH_MODE"] = WORKSHOP_AUTH_MODE
os.environ["CHAT_DEPLOYMENT_NAME"] = ""
os.environ["CHAT_MODEL_NAME"] = ""
os.environ["EMBEDDING_DEPLOYMENT_NAME"] = ""
os.environ["EMBEDDING_MODEL_NAME"] = ""
os.environ["KB_MCP_ENDPOINT"] = ""
os.environ["RESOURCE_GROUP_NAME"] = WORKSHOP_RESOURCE_GROUP
os.environ["FOUNDRY_PROJECT_NAME"] = ""

In [ ]:
import os
import shlex
import subprocess

resource_group_name = WORKSHOP_RESOURCE_GROUP
auth_mode = WORKSHOP_AUTH_MODE

cmd = [
    "bash",
    "../../scripts/assign-workshop-env.sh",
    "--resource-group",
    resource_group_name,
    "--auth-mode",
    auth_mode,
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stderr.strip():
    print(result.stderr.strip())

for line in result.stdout.splitlines():
    line = line.strip()
    if not line.startswith("export "):
        continue
    key, raw_value = line[len("export "):].split("=", 1)
    parsed = shlex.split(raw_value)
    os.environ[key] = parsed[0] if parsed else ""

print("Workshop environment variables loaded into notebook kernel. Azure token auth uses AzureCliCredential via az login.")

In [ ]:
import importlib
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

import workshop_bootstrap
importlib.reload(workshop_bootstrap)
build_workshop_config = workshop_bootstrap.build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "foundry_project_api_key": "",
    "search_service_name": "",
    "search_api_key": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config(CONFIG_OVERRIDES)
config.show()

# Workshop 4: Multi-Agent Sequential Pattern

This notebook mirrors docs/multi-agent.md and creates:
- ask-question agent
- get-answer agent
- sequential workflow YAML artifact

In [ ]:
# Uncomment this cell in a clean kernel.
# %pip install --quiet "azure-ai-projects>=2.0.0" azure-identity

In [ ]:
import os
from pathlib import Path

from azure.ai.projects.models import PromptAgentDefinition

from workshop_bootstrap import build_project_client, write_text

if not config.foundry_project_endpoint:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT or provide foundry_project_endpoint in CONFIG_OVERRIDES.")

CHAT_DEPLOYMENT_NAME = os.getenv("CHAT_DEPLOYMENT_NAME", "").strip()
if not CHAT_DEPLOYMENT_NAME:
    raise ValueError("Set CHAT_DEPLOYMENT_NAME in your environment.")

ASK_QUESTION_AGENT_NAME = "ask-question"
GET_ANSWER_AGENT_NAME = "get-answer"
WORKFLOW_NAME = "ask-question-get-answer"
WORKFLOW_FILE = Path("../Agent/ask-question-get-answer.workflow.yaml").resolve()

project = build_project_client(config)

In [ ]:
ask_question_agent = project.agents.create_version(
    agent_name=ASK_QUESTION_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_DEPLOYMENT_NAME,
        instructions="Based on the user's provided ask a question on that topic.",
        tools=[],
    ),
    description="Sequential workflow starter agent.",
)

get_answer_agent = project.agents.create_version(
    agent_name=GET_ANSWER_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_DEPLOYMENT_NAME,
        instructions="Using all tools available answer the question provided.",
        tools=[],
    ),
    description="Sequential workflow answer agent.",
)

print("Created agent versions:")
print("- ask-question:", ask_question_agent.version)
print("- get-answer:  ", get_answer_agent.version)

In [ ]:
workflow_yaml = f"""
kind: workflow
name: {WORKFLOW_NAME}
description: Sequential workshop workflow
trigger:
  kind: OnConversationStart
  id: trigger_wf
  actions:
    - kind: InvokeAzureAgent
      id: ask_question_step
      agent:
        name: {ASK_QUESTION_AGENT_NAME}
      conversationId: =System.ConversationId
      input:
        messages: =System.LastMessage.Text
      output:
        autoSend: false
        messages: Local.AskQuestionOutput
    - kind: InvokeAzureAgent
      id: get_answer_step
      agent:
        name: {GET_ANSWER_AGENT_NAME}
      conversationId: =System.ConversationId
      input:
        messages: =Local.AskQuestionOutput
      output:
        autoSend: true
    - kind: EndConversation
      id: end_conversation
""".strip()

write_text(WORKFLOW_FILE, workflow_yaml)
print(f"Workflow YAML saved to: {WORKFLOW_FILE}")

## Preview Prompt

Use this prompt in workflow preview: The topic is Dogs.

If your Foundry portal does not support workflow-file import yet, copy this YAML into the workflow designer YAML view.